# 32 - PF-020 a PF-025 - Backend Spring Boot real y persistencia mínima

Este notebook genera el scaffold del backend de producto en Spring Boot para consumir el servicio Python FastAPI validado en PF-012 a PF-019.

**Tickets cubiertos**

- PF-020: crear proyecto backend real o integrar scaffold.
- PF-021: DTOs definitivos del contrato IA.
- PF-022: endpoint backend `/api/ai/agent/report`.
- PF-023: endpoint backend de ejecución de caso.
- PF-024: modelo de datos mínimo.
- PF-025: persistencia simple.

El objetivo no es ejecutar Java dentro de Colab, sino dejar una estructura Spring Boot consistente, versionable, documentada y lista para continuar en entorno local/Codex/IDE.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, textwrap, os, re
from datetime import datetime
import pandas as pd

PFI_ROOT = Path('/content/drive/MyDrive/PFI_MVP')
REPO_ROOT = PFI_ROOT / 'repo'
RESULTS_ROOT = PFI_ROOT / 'results' / 'PF020_PF025_spring_backend_product'
DOCS_ROOT = REPO_ROOT / 'docs'
BACKLOG_ROOT = REPO_ROOT / 'backlogProducto'
BACKEND_ROOT = REPO_ROOT / 'backend_product_spring'

for p in [RESULTS_ROOT, DOCS_ROOT, BACKLOG_ROOT, BACKEND_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print('PFI_ROOT:', PFI_ROOT)
print('REPO_ROOT:', REPO_ROOT, '| exists:', REPO_ROOT.exists())
print('BACKEND_ROOT:', BACKEND_ROOT)
print('RESULTS_ROOT:', RESULTS_ROOT)


Mounted at /content/drive
PFI_ROOT: /content/drive/MyDrive/PFI_MVP
REPO_ROOT: /content/drive/MyDrive/PFI_MVP/repo | exists: True
BACKEND_ROOT: /content/drive/MyDrive/PFI_MVP/repo/backend_product_spring
RESULTS_ROOT: /content/drive/MyDrive/PFI_MVP/results/PF020_PF025_spring_backend_product


## 1. Verificación de entradas

In [2]:
inputs = {
    'pf012_docs': REPO_ROOT / 'docs/PF012_PF019_service_endpoints_agent.md',
    'python_api': REPO_ROOT / 'ai_service/pfi_ai_service/api.py',
    'model_registry_final': REPO_ROOT / 'config/model_registry_final.json',
    'pf012_report_hint': PFI_ROOT / 'results/PF012_PF019_service_endpoints_agent/PF012_PF019_report.json',
}

for name, path in inputs.items():
    print(name, '->', path, '| exists:', path.exists())


pf012_docs -> /content/drive/MyDrive/PFI_MVP/repo/docs/PF012_PF019_service_endpoints_agent.md | exists: True
python_api -> /content/drive/MyDrive/PFI_MVP/repo/ai_service/pfi_ai_service/api.py | exists: True
model_registry_final -> /content/drive/MyDrive/PFI_MVP/repo/config/model_registry_final.json | exists: True
pf012_report_hint -> /content/drive/MyDrive/PFI_MVP/results/PF012_PF019_service_endpoints_agent/PF012_PF019_report.json | exists: True


## 2. Generación del backend Spring Boot

In [3]:

def write_file(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).strip() + "\n", encoding='utf-8')
    print('Wrote:', path)

write_file(BACKEND_ROOT / 'pom.xml', '''
<project xmlns="http://maven.apache.org/POM/4.0.0"
         xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
         xsi:schemaLocation="http://maven.apache.org/POM/4.0.0 https://maven.apache.org/xsd/maven-4.0.0.xsd">
    <modelVersion>4.0.0</modelVersion>
    <groupId>ar.edu.uade.pfi</groupId>
    <artifactId>backend-product-spring</artifactId>
    <version>0.1.0-SNAPSHOT</version>
    <name>PFI Backend Product Spring</name>
    <description>Backend Spring Boot para consumir el servicio Python IA de RM lumbar</description>
    <properties>
        <java.version>17</java.version>
        <spring.boot.version>3.3.5</spring.boot.version>
    </properties>
    <dependencyManagement>
        <dependencies>
            <dependency>
                <groupId>org.springframework.boot</groupId>
                <artifactId>spring-boot-dependencies</artifactId>
                <version>${spring.boot.version}</version>
                <type>pom</type>
                <scope>import</scope>
            </dependency>
        </dependencies>
    </dependencyManagement>
    <dependencies>
        <dependency><groupId>org.springframework.boot</groupId><artifactId>spring-boot-starter-web</artifactId></dependency>
        <dependency><groupId>org.springframework.boot</groupId><artifactId>spring-boot-starter-webflux</artifactId></dependency>
        <dependency><groupId>org.springframework.boot</groupId><artifactId>spring-boot-starter-validation</artifactId></dependency>
        <dependency><groupId>org.springframework.boot</groupId><artifactId>spring-boot-starter-actuator</artifactId></dependency>
        <dependency><groupId>org.springframework.boot</groupId><artifactId>spring-boot-starter-test</artifactId><scope>test</scope></dependency>
    </dependencies>
    <build>
        <plugins>
            <plugin><groupId>org.springframework.boot</groupId><artifactId>spring-boot-maven-plugin</artifactId><version>${spring.boot.version}</version></plugin>
            <plugin><groupId>org.apache.maven.plugins</groupId><artifactId>maven-compiler-plugin</artifactId><version>3.13.0</version><configuration><source>${java.version}</source><target>${java.version}</target></configuration></plugin>
        </plugins>
    </build>
</project>
''')

write_file(BACKEND_ROOT / 'src/main/resources/application.yml', '''
server:
  port: 8080
spring:
  application:
    name: pfi-backend-product-spring
management:
  endpoints:
    web:
      exposure:
        include: health,info
pfi:
  ai-service:
    base-url: ${PFI_AI_SERVICE_URL:http://localhost:8000}
    timeout-seconds: 60
  product:
    human-review-required: true
    not-clinical-diagnosis: true
''')

write_file(BACKEND_ROOT / 'README.md', '''
# Backend Product Spring - PFI RM Lumbar

Backend Spring Boot del producto final para consumir el servicio Python FastAPI generado en PF-012 a PF-019.

## Responsabilidad

- Orquestar llamadas al servicio Python IA.
- Exponer contrato REST al frontend.
- Mantener estado de revisión profesional.
- Persistir de forma mínima reportes y revisiones.
- Preservar el enfoque human-in-the-loop.

## Endpoints propuestos

- GET /api/ai/health
- GET /api/ai/models
- POST /api/ai/pipeline/run
- GET /api/ai/agent/report/{runId}
- PATCH /api/ai/review/{runId}

## Ejecución local

```bash
cd backend_product_spring
mvn spring-boot:run
```

Variable opcional:

```bash
export PFI_AI_SERVICE_URL=http://localhost:8000
```

## Alcance metodológico

El backend no emite diagnóstico clínico. Consume una respuesta técnica de apoyo, registra estado de revisión y mantiene la validación profesional como condición de uso.
''')


Wrote: /content/drive/MyDrive/PFI_MVP/repo/backend_product_spring/pom.xml
Wrote: /content/drive/MyDrive/PFI_MVP/repo/backend_product_spring/src/main/resources/application.yml
Wrote: /content/drive/MyDrive/PFI_MVP/repo/backend_product_spring/README.md


### Código Java

In [4]:

BASE_JAVA = BACKEND_ROOT / 'src/main/java/ar/edu/uade/pfi/backend'

write_file(BASE_JAVA / 'AiBackendApplication.java', '''
package ar.edu.uade.pfi.backend;

import org.springframework.boot.SpringApplication;
import org.springframework.boot.autoconfigure.SpringBootApplication;
import org.springframework.boot.context.properties.ConfigurationPropertiesScan;

@SpringBootApplication
@ConfigurationPropertiesScan
public class AiBackendApplication {
    public static void main(String[] args) {
        SpringApplication.run(AiBackendApplication.class, args);
    }
}
''')

write_file(BASE_JAVA / 'config/AiServiceProperties.java', '''
package ar.edu.uade.pfi.backend.config;

import org.springframework.boot.context.properties.ConfigurationProperties;

@ConfigurationProperties(prefix = "pfi.ai-service")
public record AiServiceProperties(String baseUrl, Integer timeoutSeconds) {
    public int resolvedTimeoutSeconds() { return timeoutSeconds == null ? 60 : timeoutSeconds; }
}
''')

write_file(BASE_JAVA / 'config/WebClientConfig.java', '''
package ar.edu.uade.pfi.backend.config;

import org.springframework.context.annotation.Bean;
import org.springframework.context.annotation.Configuration;
import org.springframework.web.reactive.function.client.WebClient;

@Configuration
public class WebClientConfig {
    @Bean
    public WebClient aiServiceWebClient(AiServiceProperties properties) {
        return WebClient.builder().baseUrl(properties.baseUrl()).build();
    }
}
''')

write_file(BASE_JAVA / 'dto/AiModelDto.java', '''
package ar.edu.uade.pfi.backend.dto;

public record AiModelDto(String modelKey, String plane, String datasetKey, String status, String checkpointPath, String sha256) {}
''')

write_file(BASE_JAVA / 'dto/PipelineRunRequestDto.java', '''
package ar.edu.uade.pfi.backend.dto;

import jakarta.validation.constraints.NotBlank;
import java.util.Map;

public record PipelineRunRequestDto(@NotBlank String caseId, @NotBlank String plane, String modelKey, String inputPath, Map<String, Object> options) {}
''')

write_file(BASE_JAVA / 'dto/MeasurementDto.java', '''
package ar.edu.uade.pfi.backend.dto;

public record MeasurementDto(String className, Integer classId, Double areaPixels, Double foregroundRatio, Double bboxWidth, Double bboxHeight) {}
''')

write_file(BASE_JAVA / 'dto/AgentDecisionDto.java', '''
package ar.edu.uade.pfi.backend.dto;

import java.util.List;
import java.util.Map;

public record AgentDecisionDto(String priority, String status, List<String> flags, List<String> reasons, Boolean humanReviewRequired, Boolean notClinicalDiagnosis, Boolean notReal3dReconstruction, Map<String, Object> policy) {}
''')

write_file(BASE_JAVA / 'dto/AiRunResponseDto.java', '''
package ar.edu.uade.pfi.backend.dto;

import java.util.List;
import java.util.Map;

public record AiRunResponseDto(String runId, String caseId, String plane, String modelKey, String overlayPath, List<MeasurementDto> measurements, AgentDecisionDto agentDecision, Map<String, Object> raw) {}
''')

write_file(BASE_JAVA / 'dto/ReviewUpdateRequestDto.java', '''
package ar.edu.uade.pfi.backend.dto;

import jakarta.validation.constraints.NotBlank;

public record ReviewUpdateRequestDto(@NotBlank String reviewStatus, String reviewer, String comment) {}
''')

write_file(BASE_JAVA / 'dto/ReviewStatusDto.java', '''
package ar.edu.uade.pfi.backend.dto;

import java.time.Instant;

public record ReviewStatusDto(String runId, String reviewStatus, String reviewer, String comment, Instant updatedAt, Boolean humanReviewRequired) {}
''')

write_file(BASE_JAVA / 'client/AiServiceClient.java', '''
package ar.edu.uade.pfi.backend.client;

import ar.edu.uade.pfi.backend.dto.PipelineRunRequestDto;
import java.util.Map;
import org.springframework.stereotype.Component;
import org.springframework.web.reactive.function.client.WebClient;

@Component
public class AiServiceClient {
    private final WebClient aiServiceWebClient;
    public AiServiceClient(WebClient aiServiceWebClient) { this.aiServiceWebClient = aiServiceWebClient; }
    public Map<String, Object> health() { return aiServiceWebClient.get().uri("/health").retrieve().bodyToMono(Map.class).block(); }
    public Map<String, Object> models() { return aiServiceWebClient.get().uri("/models").retrieve().bodyToMono(Map.class).block(); }
    public Map<String, Object> runPipeline(PipelineRunRequestDto request) { return aiServiceWebClient.post().uri("/pipeline/run").bodyValue(request).retrieve().bodyToMono(Map.class).block(); }
    public Map<String, Object> agentReport(String runId) { return aiServiceWebClient.get().uri("/agent/report/{runId}", runId).retrieve().bodyToMono(Map.class).block(); }
}
''')

write_file(BASE_JAVA / 'service/ReviewStoreService.java', '''
package ar.edu.uade.pfi.backend.service;

import ar.edu.uade.pfi.backend.dto.ReviewStatusDto;
import ar.edu.uade.pfi.backend.dto.ReviewUpdateRequestDto;
import java.time.Instant;
import java.util.Map;
import java.util.concurrent.ConcurrentHashMap;
import org.springframework.stereotype.Service;

@Service
public class ReviewStoreService {
    private final Map<String, ReviewStatusDto> reviewStore = new ConcurrentHashMap<>();
    public ReviewStatusDto updateReview(String runId, ReviewUpdateRequestDto request) {
        ReviewStatusDto status = new ReviewStatusDto(runId, request.reviewStatus(), request.reviewer(), request.comment(), Instant.now(), true);
        reviewStore.put(runId, status);
        return status;
    }
    public ReviewStatusDto findOrDefault(String runId) {
        return reviewStore.getOrDefault(runId, new ReviewStatusDto(runId, "pendiente_revision_profesional", null, null, null, true));
    }
}
''')

write_file(BASE_JAVA / 'service/AiBackendService.java', '''
package ar.edu.uade.pfi.backend.service;

import ar.edu.uade.pfi.backend.client.AiServiceClient;
import ar.edu.uade.pfi.backend.dto.PipelineRunRequestDto;
import java.util.LinkedHashMap;
import java.util.Map;
import org.springframework.stereotype.Service;

@Service
public class AiBackendService {
    private final AiServiceClient aiServiceClient;
    private final ReviewStoreService reviewStoreService;
    public AiBackendService(AiServiceClient aiServiceClient, ReviewStoreService reviewStoreService) {
        this.aiServiceClient = aiServiceClient;
        this.reviewStoreService = reviewStoreService;
    }
    public Map<String, Object> health() {
        Map<String, Object> response = new LinkedHashMap<>();
        response.put("backendStatus", "ok");
        response.put("humanReviewRequired", true);
        response.put("notClinicalDiagnosis", true);
        response.put("pythonService", aiServiceClient.health());
        return response;
    }
    public Map<String, Object> models() { return aiServiceClient.models(); }
    public Map<String, Object> runPipeline(PipelineRunRequestDto request) {
        Map<String, Object> run = aiServiceClient.runPipeline(request);
        run.put("backendHumanReviewRequired", true);
        run.put("backendNotClinicalDiagnosis", true);
        return run;
    }
    public Map<String, Object> agentReport(String runId) {
        Map<String, Object> report = aiServiceClient.agentReport(runId);
        report.put("review", reviewStoreService.findOrDefault(runId));
        report.put("humanReviewRequired", true);
        report.put("notClinicalDiagnosis", true);
        return report;
    }
}
''')

write_file(BASE_JAVA / 'controller/AiBackendController.java', '''
package ar.edu.uade.pfi.backend.controller;

import ar.edu.uade.pfi.backend.dto.PipelineRunRequestDto;
import ar.edu.uade.pfi.backend.dto.ReviewStatusDto;
import ar.edu.uade.pfi.backend.dto.ReviewUpdateRequestDto;
import ar.edu.uade.pfi.backend.service.AiBackendService;
import ar.edu.uade.pfi.backend.service.ReviewStoreService;
import jakarta.validation.Valid;
import java.util.Map;
import org.springframework.web.bind.annotation.*;

@RestController
@RequestMapping("/api/ai")
public class AiBackendController {
    private final AiBackendService aiBackendService;
    private final ReviewStoreService reviewStoreService;
    public AiBackendController(AiBackendService aiBackendService, ReviewStoreService reviewStoreService) {
        this.aiBackendService = aiBackendService;
        this.reviewStoreService = reviewStoreService;
    }
    @GetMapping("/health")
    public Map<String, Object> health() { return aiBackendService.health(); }
    @GetMapping("/models")
    public Map<String, Object> models() { return aiBackendService.models(); }
    @PostMapping("/pipeline/run")
    public Map<String, Object> runPipeline(@Valid @RequestBody PipelineRunRequestDto request) { return aiBackendService.runPipeline(request); }
    @GetMapping("/agent/report/{runId}")
    public Map<String, Object> agentReport(@PathVariable String runId) { return aiBackendService.agentReport(runId); }
    @PatchMapping("/review/{runId}")
    public ReviewStatusDto updateReview(@PathVariable String runId, @Valid @RequestBody ReviewUpdateRequestDto request) { return reviewStoreService.updateReview(runId, request); }
}
''')


Wrote: /content/drive/MyDrive/PFI_MVP/repo/backend_product_spring/src/main/java/ar/edu/uade/pfi/backend/AiBackendApplication.java
Wrote: /content/drive/MyDrive/PFI_MVP/repo/backend_product_spring/src/main/java/ar/edu/uade/pfi/backend/config/AiServiceProperties.java
Wrote: /content/drive/MyDrive/PFI_MVP/repo/backend_product_spring/src/main/java/ar/edu/uade/pfi/backend/config/WebClientConfig.java
Wrote: /content/drive/MyDrive/PFI_MVP/repo/backend_product_spring/src/main/java/ar/edu/uade/pfi/backend/dto/AiModelDto.java
Wrote: /content/drive/MyDrive/PFI_MVP/repo/backend_product_spring/src/main/java/ar/edu/uade/pfi/backend/dto/PipelineRunRequestDto.java
Wrote: /content/drive/MyDrive/PFI_MVP/repo/backend_product_spring/src/main/java/ar/edu/uade/pfi/backend/dto/MeasurementDto.java
Wrote: /content/drive/MyDrive/PFI_MVP/repo/backend_product_spring/src/main/java/ar/edu/uade/pfi/backend/dto/AgentDecisionDto.java
Wrote: /content/drive/MyDrive/PFI_MVP/repo/backend_product_spring/src/main/java/ar/ed

## 3. Documentación del contrato backend

In [5]:

write_file(DOCS_ROOT / 'PF020_PF025_spring_backend_contract.md', '''
# PF-020 a PF-025 - Contrato backend Spring Boot

## Propósito

Este documento define la capa backend Spring Boot que consume el servicio Python FastAPI y expone una API estable para el frontend del MVP.

## Cadena de integración

Frontend React -> Spring Boot backend -> Python FastAPI -> modelos IA -> agente -> reporte revisable.

## Endpoints backend

| Método | Endpoint | Uso |
|---|---|---|
| GET | `/api/ai/health` | Verificar backend y servicio Python. |
| GET | `/api/ai/models` | Consultar modelos finales registrados. |
| POST | `/api/ai/pipeline/run` | Ejecutar inferencia/pipeline sobre un caso. |
| GET | `/api/ai/agent/report/{runId}` | Recuperar reporte del agente y estado de revisión. |
| PATCH | `/api/ai/review/{runId}` | Actualizar estado de revisión profesional. |

## Estados de revisión sugeridos

- `pendiente_revision_profesional`
- `aceptado`
- `observado`
- `corregido`
- `descartado`

## Política metodológica

El backend preserva `humanReviewRequired=true` y `notClinicalDiagnosis=true`. La salida es una ayuda técnica revisable, no un diagnóstico clínico automático.
''')

write_file(DOCS_ROOT / 'PF020_PF025_frontend_backend_contract.md', '''
# PF-020 a PF-025 - Contrato frontend/backend

## Datos mínimos para el frontend

El frontend debe poder consumir:

- `runId`
- `caseId`
- `plane`
- `modelKey`
- `overlayPath`
- `measurements`
- `agentDecision.priority`
- `agentDecision.flags`
- `agentDecision.reasons`
- `humanReviewRequired`
- `review.reviewStatus`

## Acciones mínimas de UI

- Ejecutar pipeline sobre caso de prueba.
- Ver worklist/reporte del agente.
- Mostrar prioridad y flags.
- Mostrar overlay o ruta de evidencia visual.
- Cambiar estado de revisión.
- Agregar comentario profesional.

## Límite

No se debe mostrar la salida como diagnóstico final. La interfaz debe usar lenguaje de apoyo, revisión y evidencia.
''')


Wrote: /content/drive/MyDrive/PFI_MVP/repo/docs/PF020_PF025_spring_backend_contract.md
Wrote: /content/drive/MyDrive/PFI_MVP/repo/docs/PF020_PF025_frontend_backend_contract.md


## 4. Checks estáticos

In [6]:

required_files = [
    'pom.xml', 'README.md', 'src/main/resources/application.yml',
    'src/main/java/ar/edu/uade/pfi/backend/AiBackendApplication.java',
    'src/main/java/ar/edu/uade/pfi/backend/config/AiServiceProperties.java',
    'src/main/java/ar/edu/uade/pfi/backend/config/WebClientConfig.java',
    'src/main/java/ar/edu/uade/pfi/backend/client/AiServiceClient.java',
    'src/main/java/ar/edu/uade/pfi/backend/service/AiBackendService.java',
    'src/main/java/ar/edu/uade/pfi/backend/service/ReviewStoreService.java',
    'src/main/java/ar/edu/uade/pfi/backend/controller/AiBackendController.java',
    'src/main/java/ar/edu/uade/pfi/backend/dto/PipelineRunRequestDto.java',
    'src/main/java/ar/edu/uade/pfi/backend/dto/ReviewUpdateRequestDto.java',
    'src/main/java/ar/edu/uade/pfi/backend/dto/ReviewStatusDto.java',
]

inventory=[]
for rel in required_files:
    p=BACKEND_ROOT/rel
    inventory.append({'relative_path': f'backend_product_spring/{rel}', 'exists': p.exists(), 'size_bytes': p.stat().st_size if p.exists() else 0})
inventory_df=pd.DataFrame(inventory)
inventory_csv=RESULTS_ROOT/'PF020_PF025_backend_files_inventory.csv'
inventory_df.to_csv(inventory_csv,index=False)
display(inventory_df)
print('Inventory:', inventory_csv)

controller_text=(BASE_JAVA/'controller/AiBackendController.java').read_text(encoding='utf-8')
service_text=(BASE_JAVA/'service/AiBackendService.java').read_text(encoding='utf-8')
client_text=(BASE_JAVA/'client/AiServiceClient.java').read_text(encoding='utf-8')

checks=[
    {'check':'backend_root_exists','ok':BACKEND_ROOT.exists(),'detail':str(BACKEND_ROOT)},
    {'check':'required_files_exist','ok':bool(inventory_df['exists'].all()),'detail':f"{inventory_df['exists'].sum()}/{len(inventory_df)}"},
    {'check':'has_health_endpoint','ok':'/health' in controller_text,'detail':'GET /api/ai/health'},
    {'check':'has_models_endpoint','ok':'/models' in controller_text,'detail':'GET /api/ai/models'},
    {'check':'has_pipeline_endpoint','ok':'/pipeline/run' in controller_text,'detail':'POST /api/ai/pipeline/run'},
    {'check':'has_agent_report_endpoint','ok':'/agent/report/{runId}' in controller_text,'detail':'GET /api/ai/agent/report/{runId}'},
    {'check':'has_review_patch_endpoint','ok':'/review/{runId}' in controller_text,'detail':'PATCH /api/ai/review/{runId}'},
    {'check':'uses_webclient_to_python_service','ok':'WebClient' in client_text and '/pipeline/run' in client_text,'detail':'AiServiceClient'},
    {'check':'preserves_human_review_policy','ok':'humanReviewRequired' in service_text and 'notClinicalDiagnosis' in service_text,'detail':'service response flags'},
    {'check':'docs_backend_contract_written','ok':(DOCS_ROOT/'PF020_PF025_spring_backend_contract.md').exists(),'detail':str(DOCS_ROOT/'PF020_PF025_spring_backend_contract.md')},
    {'check':'docs_frontend_contract_written','ok':(DOCS_ROOT/'PF020_PF025_frontend_backend_contract.md').exists(),'detail':str(DOCS_ROOT/'PF020_PF025_frontend_backend_contract.md')},
]
checks_df=pd.DataFrame(checks)
checks_csv=RESULTS_ROOT/'PF020_PF025_checks.csv'
checks_df.to_csv(checks_csv,index=False)
display(checks_df)
print('Checks:',checks_csv)
print('All OK:',bool(checks_df['ok'].all()))


,relative_path,exists,size_bytes
0,backend_product_spring/pom.xml,True,2251
1,backend_product_spring/README.md,True,916
2,backend_product_spring/src/main/resources/appl...,True,337
3,backend_product_spring/src/main/java/ar/edu/ua...,True,441
4,backend_product_spring/src/main/java/ar/edu/ua...,True,345
5,backend_product_spring/src/main/java/ar/edu/ua...,True,434
6,backend_product_spring/src/main/java/ar/edu/ua...,True,1067
7,backend_product_spring/src/main/java/ar/edu/ua...,True,1655
8,backend_product_spring/src/main/java/ar/edu/ua...,True,928
9,backend_product_spring/src/main/java/ar/edu/ua...,True,1608


Inventory: /content/drive/MyDrive/PFI_MVP/results/PF020_PF025_spring_backend_product/PF020_PF025_backend_files_inventory.csv


,check,ok,detail
0,backend_root_exists,True,/content/drive/MyDrive/PFI_MVP/repo/backend_pr...
1,required_files_exist,True,13/13
2,has_health_endpoint,True,GET /api/ai/health
3,has_models_endpoint,True,GET /api/ai/models
4,has_pipeline_endpoint,True,POST /api/ai/pipeline/run
5,has_agent_report_endpoint,True,GET /api/ai/agent/report/{runId}
6,has_review_patch_endpoint,True,PATCH /api/ai/review/{runId}
7,uses_webclient_to_python_service,True,AiServiceClient
8,preserves_human_review_policy,True,service response flags
9,docs_backend_contract_written,True,/content/drive/MyDrive/PFI_MVP/repo/docs/PF020...


Checks: /content/drive/MyDrive/PFI_MVP/results/PF020_PF025_spring_backend_product/PF020_PF025_checks.csv
All OK: True


## 5. Reporte y cierre

In [7]:

report={
    'notebook':'32_PF020_PF025_spring_backend_product',
    'goal':'generate Spring Boot product backend to consume Python AI service and expose frontend contract',
    'timestamp':datetime.now().isoformat(timespec='seconds'),
    'inputs':{k:str(v) for k,v in inputs.items()},
    'outputs':{
        'backend_root':str(BACKEND_ROOT),
        'backend_files_inventory_csv':str(inventory_csv),
        'checks_csv':str(checks_csv),
        'docs_backend_contract':str(DOCS_ROOT/'PF020_PF025_spring_backend_contract.md'),
        'docs_frontend_contract':str(DOCS_ROOT/'PF020_PF025_frontend_backend_contract.md'),
    },
    'summary':{
        'generated_backend_files':int(inventory_df['exists'].sum()),
        'all_static_checks_ok':bool(checks_df['ok'].all()),
        'backend_endpoints':['GET /api/ai/health','GET /api/ai/models','POST /api/ai/pipeline/run','GET /api/ai/agent/report/{runId}','PATCH /api/ai/review/{runId}'],
        'persistence_strategy':'in_memory_review_store_ready_to_replace_with_H2_PostgreSQL',
    },
    'methodological_policy':{
        'spring_boot_product_backend':True,
        'python_service_runs_ai':True,
        'frontend_consumes_spring_boot':True,
        'human_review_required':True,
        'not_clinical_diagnosis':True,
        'not_real_3d_reconstruction':True,
    },
    'decision':'PF020_PF025_spring_backend_ready_for_frontend_integration' if bool(checks_df['ok'].all()) else 'PF020_PF025_requires_fix',
}
report_path=RESULTS_ROOT/'PF020_PF025_report.json'
report_path.write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
print('Report:',report_path)
print(json.dumps(report,indent=2,ensure_ascii=False))

closure=f'''
# Cierre PF-020 a PF-025 - Backend Spring Boot real

Estado: cerrado si `all_static_checks_ok=true`.

## Resultado

Decision: `{report['decision']}`

## Archivos principales

- `backend_product_spring/pom.xml`
- `backend_product_spring/src/main/java/ar/edu/uade/pfi/backend/AiBackendApplication.java`
- `backend_product_spring/src/main/java/ar/edu/uade/pfi/backend/client/AiServiceClient.java`
- `backend_product_spring/src/main/java/ar/edu/uade/pfi/backend/service/AiBackendService.java`
- `backend_product_spring/src/main/java/ar/edu/uade/pfi/backend/service/ReviewStoreService.java`
- `backend_product_spring/src/main/java/ar/edu/uade/pfi/backend/controller/AiBackendController.java`

## Endpoints backend

- `GET /api/ai/health`
- `GET /api/ai/models`
- `POST /api/ai/pipeline/run`
- `GET /api/ai/agent/report/{{runId}}`
- `PATCH /api/ai/review/{{runId}}`

## Politica metodologica

- Spring Boot funciona como backend de producto.
- Python/FastAPI mantiene la ejecucion de IA.
- El frontend debe consumir Spring Boot, no directamente Python.
- La salida mantiene revision humana obligatoria y no diagnostico clinico.

## Proximo bloque

PF-026 a PF-030: frontend final, vistas requeridas, integracion con backend y estados de revision.
'''
closure_path=BACKLOG_ROOT/'PF020_PF025_resultados_cierre.md'
closure_path.write_text(textwrap.dedent(closure).strip()+'\n',encoding='utf-8')
print('Closure:',closure_path)


Report: /content/drive/MyDrive/PFI_MVP/results/PF020_PF025_spring_backend_product/PF020_PF025_report.json
{
  "notebook": "32_PF020_PF025_spring_backend_product",
  "goal": "generate Spring Boot product backend to consume Python AI service and expose frontend contract",
  "timestamp": "2026-06-30T22:42:02",
  "inputs": {
    "pf012_docs": "/content/drive/MyDrive/PFI_MVP/repo/docs/PF012_PF019_service_endpoints_agent.md",
    "python_api": "/content/drive/MyDrive/PFI_MVP/repo/ai_service/pfi_ai_service/api.py",
    "model_registry_final": "/content/drive/MyDrive/PFI_MVP/repo/config/model_registry_final.json",
    "pf012_report_hint": "/content/drive/MyDrive/PFI_MVP/results/PF012_PF019_service_endpoints_agent/PF012_PF019_report.json"
  },
  "outputs": {
    "backend_root": "/content/drive/MyDrive/PFI_MVP/repo/backend_product_spring",
    "backend_files_inventory_csv": "/content/drive/MyDrive/PFI_MVP/results/PF020_PF025_spring_backend_product/PF020_PF025_backend_files_inventory.csv",
    "c